# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [9]:
#Definition of the tables.
import pandas as pd

data = {
    'table': [
        'artists', 
        'albums', 
        'songs', 
        'publishers', 
        'lyrics'
    ],
    'definition': [
        'Information about music artists, including names and genres.',
        'Details about music albums: titles, release dates, and artist IDs.',
        'Individual track information, including song titles and durations.',
        'Data about record labels and music publishing companies.',
        'Full text of song lyrics linked to specific song IDs.'
    ]
}

df = pd.DataFrame(data)
print(df)

        table                                         definition
0     artists  Information about music artists, including nam...
1      albums  Details about music albums: titles, release da...
2       songs  Individual track information, including song t...
3  publishers  Data about record labels and music publishing ...
4      lyrics  Full text of song lyrics linked to specific so...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [11]:
print(text_tables)

artists: Information about music artists, including names and genres.
albums: Details about music albums: titles, release dates, and artist IDs.
songs: Individual track information, including song titles and durations.
publishers: Data about record labels and music publishing companies.
lyrics: Full text of song lyrics linked to specific song IDs.


In [12]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [15]:

pqt1 = prompt_question_tables.format(
    tables=text_tables, 
    question="Which songs were published by 'Universal Music Group'?"
)
print(pqt1)


Given the following tables and their content definitions,
###Tables
artists: Information about music artists, including names and genres.
albums: Details about music albums: titles, release dates, and artist IDs.
songs: Individual track information, including song titles and durations.
publishers: Data about record labels and music publishing companies.
lyrics: Full text of song lyrics linked to specific song IDs.

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
Which songs were published by 'Universal Music Group'?



In [16]:
print(return_OAI(pqt1))

```json
{
    "tables": ["songs", "publishers"]
}
```


In [18]:
pqt2 = prompt_question_tables.format(
    tables=text_tables, 
    question="How many albums and in which years did each artist release? Sort from most to least, including those who released nothing."
)


In [21]:
print(return_OAI(pqt2))

{
    "tables": ["artists", "albums"]
}


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [22]:
question_3 = "Which publisher is the biggest?"

pqt3 = prompt_question_tables.format(
    tables=text_tables, 
    question=question_3
)

response_3 = return_OAI(pqt3)
print(f"User Question: {question_3}")
print(f"Model Response: {response_3}")

User Question: Which publisher is the biggest?
Model Response: ```json
{
    "tables": ["publishers"]
}
```


In [23]:
question_4 = "List all songs and their durations from the album 'The Bridge' by the artist 'Sting'."

pqt4 = prompt_question_tables.format(
    tables=text_tables, 
    question=question_4
)

response_4 = return_OAI(pqt4)
print(f"User Question: {question_4}")
print(f"Model Response: {response_4}")

User Question: List all songs and their durations from the album 'The Bridge' by the artist 'Sting'.
Model Response: {
    "tables": ["songs", "albums", "artists"]
}


In [24]:
question_5 = "Find the full lyrics for all songs on the album 'Midnights' that are longer than 4 minutes."

pqt5 = prompt_question_tables.format(
    tables=text_tables, 
    question=question_5
)
response_5 = return_OAI(pqt5)
print(f"User Question: {question_5}")
print(f"Model Response: {response_5}")

User Question: Find the full lyrics for all songs on the album 'Midnights' that are longer than 4 minutes.
Model Response: ```json
{
    "tables": ["albums", "songs", "lyrics"]
}
```


Report Summary 
Project Overview:
The goal of this exercise was to evaluate the ability of the gpt-3.5-turbo model to perform Schema Linking. Specifically, I tested if the model could correctly identify the necessary database tables required to answer natural language questions based only on table names and short descriptions.

Key Findings:

Simple Queries: The model excelled at one-table lookups (e.g., finding artists or publishers).

Multi-table Joins: For complex questions involving lyrics and specific albums, the model successfully identified the relationships between artists, albums, songs, and lyrics.

Ambiguity Handling: When asked "which is the biggest," the model smartly included related tables to allow for counting/aggregation, rather than just looking at the metadata.


What I Learned:
I learned that Large Language Models (LLMs) can act as an effective intermediary layer between a user and a database. By using a two-step process—Table Selection followed by SQL Generation—we can build more reliable "Text-to-SQL" systems. Even with limited local hardware (like a Mac Air M1), using APIs allows us to process complex data logic efficiently.